## Step 0: Setup

In [17]:
import requests
import pandas as pd
from datetime import datetime

## Step 1: Extract using API call

In [18]:
def extract_weather():
    url = "https://api.open-meteo.com/v1/forecast"

    params = {
        "latitude": 18.52,   # Pune
        "longitude": 73.85,
        "current_weather": True
    }
    response = requests.get(url, params=params)
    data = response.json()
    return data

In [19]:
data = extract_weather()
data

{'latitude': 18.5,
 'longitude': 73.875,
 'generationtime_ms': 0.049591064453125,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 542.0,
 'current_weather_units': {'time': 'iso8601',
  'interval': 'seconds',
  'temperature': '°C',
  'windspeed': 'km/h',
  'winddirection': '°',
  'is_day': '',
  'weathercode': 'wmo code'},
 'current_weather': {'time': '2026-04-18T14:00',
  'interval': 900,
  'temperature': 33.1,
  'windspeed': 6.4,
  'winddirection': 286,
  'is_day': 0,
  'weathercode': 2}}

## Step 2: Transform and clean data

In [20]:
#converting JSON to pandas dataframe
def transform_weather(data):
    current = data.get("current_weather",{})
    df = pd.DataFrame([current])
    df["ingestion_time"]=datetime.now()
    return df

In [21]:
df = transform_weather(data)
df

,time,interval,temperature,windspeed,winddirection,is_day,weathercode,ingestion_time
0,2026-04-18T14:00,900,33.1,6.4,286,0,2,2026-04-18 19:42:33.441200


In [22]:
import logging

logging.basicConfig(
    filename="pipeline.log",
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)

## Step 3: Load(Saving to csv)

In [23]:
def load_weather(df):
    file_name = "pune_weather.csv"
    df.to_csv(file_name, mode ='a', index=False, header=not pd.io.common.file_exists(file_name))
    logging.info("Data saved to csv")

In [24]:
load_weather(df)

## Step 4: Add Error handling 

In [25]:
def run_pipeline():
    try:
        data = extract_weather()
        df= transform_weather(data)
        load_weather(df)
        logging.info("Pipeline executed successfully")

    except EException as e:
        logging.info("Error occurred:", e)

In [26]:
run_pipeline()

## Step 5: Simulate Scheduling

In [27]:
import time

for i in range(3):
    run_pipeline()
    time.sleep(10) #to run every 10 secs

## SQL import

In [28]:
import sqlite3
conn =sqlite3.connect("weather.db")
df.to_sql("weather", conn, if_exists="append",index=False)

1

In [29]:
query = "SELECT * from weather ORDER BY ingestion_time DESC"
pd.read_sql(query, conn)

,time,interval,temperature,windspeed,winddirection,is_day,weathercode,ingestion_time
0,2026-04-18T14:00,900,33.1,6.4,286,0,2,2026-04-18 19:42:33.441200
1,2026-04-18T13:00,900,35.2,7.5,305,1,2,2026-04-18 18:48:48.628319
